# C1-ml-fundamentals — Practice p22 — Solution

**Challenge.** The biggest-gap rule is not the last word in 1-D clustering.
Consider the unlabeled dataset below: two tight groups of readings — plus one
extreme stray value at 15.0.

**(a)** Apply the lesson's biggest-gap rule (sort, cut at the largest
neighbour gap). Print the two groups and observe the failure: the "cluster"
on one side is a single stray point.

**(b)** Implement a sturdier split:

```python
def best_two_group_split(values):
    ...
```

Sort the values; for **every** possible cut position k (1 ≤ k ≤ n−1) form
the left group `s[:k]` and right group `s[k:]`, and score the cut by *total
within-group spread*: the sum over both groups of `np.sum(np.abs(group -
group.mean()))`. Return `(group_a, group_b, cost)` for the cut with the
smallest total spread (`group_a` = lower values, `cost` = that smallest
total spread as a float). A `for` loop over cut positions is allowed; the
per-group arithmetic must be array operations.

**(c)** Apply it to the same data, print the result, and explain in the
markdown cell why minimizing within-group spread resists the stray point
that fooled the gap rule.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
values = np.array([2.1, 2.4, 2.5, 2.8, 3.0, 7.9, 8.1, 8.4, 8.6, 15.0])

In [ ]:
# (a) the biggest-gap rule gets distracted by the stray point
s = np.sort(values)
k_gap = int(np.argmax(np.diff(s)))
print("gap rule:", s[:k_gap + 1], "|", s[k_gap + 1:])

In [ ]:
# (b) try every cut, keep the one with least total within-group spread
def best_two_group_split(values):
    s = np.sort(values)
    best_k, best_cost = 1, np.inf
    for k in range(1, len(s)):
        left, right = s[:k], s[k:]
        cost = np.sum(np.abs(left - left.mean())) + np.sum(np.abs(right - right.mean()))
        if cost < best_cost:
            best_k, best_cost = k, float(cost)
    return s[:best_k], s[best_k:], best_cost

In [ ]:
# (c) the sturdier split
group_a, group_b, cost = best_two_group_split(values)
print("spread rule:", group_a, "|", group_b, f"  cost = {cost:.2f}")

**(c) explanation.** The gap rule looks at exactly one number — the largest
neighbour gap — and the stray point at 15.0 owns that gap, so the rule
"clusters" the data into nine points versus one stray. The spread score
instead judges every candidate cut by how internally tight *both* groups
would be, summed over all points. Cutting between 3.0 and 7.9 leaves two
genuinely tight groups (the stray inflates the right group's spread a
little, but far less than any other cut inflates the total), so the
minimum-spread cut recovers the real structure. Scoring whole groups rather
than one local gap is exactly the step from a heuristic toward real
clustering methods.

### Answer check

In [ ]:
# gap rule isolates the stray point
assert len(s[k_gap + 1:]) == 1 and s[-1] == 15.0
# spread rule recovers the two real groups (stray joins the upper group)
assert np.allclose(group_a, [2.1, 2.4, 2.5, 2.8, 3.0])
assert np.allclose(group_b, [7.9, 8.1, 8.4, 8.6, 15.0])
assert np.isclose(cost, 12.159999999999998)
print("p22 OK")